In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [2]:
train_data_dir = 'imagedataset\dataset'

In [3]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range = 40, width_shift_range = 0.2, height_shift_range = 0.2, shear_range = 0.2, zoom_range = 0.2, horizontal_flip = True)

In [4]:
train_generator = train_datagen.flow_from_directory(train_data_dir, batch_size = 20, class_mode = 'binary', target_size = (224, 224))

Found 2829 images belonging to 2 classes.


In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
base_model = VGG16(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.models.Model(base_model.input, x)

early_stopping = EarlyStopping(monitor='acc', patience=5, restore_best_weights=True)

model.compile(optimizer=tf.keras.optimizers.RMSprop(lr=0.0001), 
              loss='binary_crossentropy', 
              metrics=['acc'])
inception_hist = model.fit(train_generator, 
                           steps_per_epoch=len(train_generator),
                           epochs=15,
                           callbacks=[early_stopping])

print("Final Accuracy: ", inception_hist.history['acc'][-1])

c:\Users\abhinav bhardwaj\miniconda3\envs\tl\lib\site-packages\keras\optimizers\optimizer_v2\rmsprop.py:140: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/15
142/142 [==============================] - 53s 287ms/step - loss: 0.5241 - acc: 0.7869
Epoch 2/15
142/142 [==============================] - 38s 270ms/step - loss: 0.5174 - acc: 0.7929
Epoch 3/15
142/142 [==============================] - 33s 233ms/step - loss: 0.5054 - acc: 0.7943
Epoch 4/15
142/142 [==============================] - 37s 259ms/step - loss: 0.4999 - acc: 0.7946
Epoch 5/15
142/142 [==============================] - 35s 246ms/step - loss: 0.4946 - acc: 0.7950
Epoch 6/15
142/142 [==============================] - 40s 282ms/step - loss: 0.4909 - acc: 0.7967
Epoch 7/15
142/142 [==============================] - 35s 241ms/step - loss: 0.4870 - acc: 0.7967
Epoch 8/15
142/142 [==============================] - 35s 245ms/step - loss: 0.4824 - acc: 0.7939
Epoch 9/15
142/142 [==============================] - 34s 240ms/step - loss: 0.4688 - acc: 0.7996
Epoch 10/15
142/142 [==============================] - 35s 242ms/step - loss: 0.4674 - acc: 0.7992
Epoch 11/15
142/142

In [6]:
model.save('vg16.h5')